In [1]:
import math
import torch
from main.model.downstream.clare.datamodule import ClareDataModule

dm = ClareDataModule("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream-clare", seed=1,
                     batch_size=32)

In [2]:
dl = dm.train_dataloader()
it = iter(dl)

scores_value = 0
score_count = 0
for i in it:
    if not math.isnan(i["assessment", "scores"].sum().item()):
        scores_value += (i["assessment", "scores"] >= 5).sum().item()
        score_count += i.batch_size[0]

p_train = scores_value / score_count
majority_class = int(p_train >= 0.5)

In [6]:
dl = dm.test_dataloader()
it = iter(dl)

all_y = []
for i in it:
    all_y.append((i["assessment", "scores"] >= 5).long())

y = torch.cat(all_y)

baseline_acc = (y == majority_class).float().mean()

In [7]:
baseline_acc

tensor(0.7179)

In [8]:
import torch.nn.functional as F

# Cross-entropy baseline
probs = torch.tensor([1 - p_train, p_train], dtype=torch.float32)
log_probs = probs.log().unsqueeze(0).repeat(y.shape[0], 1)
baseline_ce = F.nll_loss(log_probs, y)

In [9]:
baseline_ce

tensor(0.5954)

In [10]:
baseline_acc

tensor(0.7179)